# octlm on Colab

Runs the training experiments on a Colab GPU. Set **Runtime > Change runtime type > GPU**
before anything else, then run the cells in order.

This notebook uses Colab's preinstalled PyTorch rather than `uv sync`, because the lockfile
pins the CPU build. The Python and PyTorch versions therefore differ from the local
environment. Timings from Colab are not comparable with the CPU timings in `notes/`; loss
and bits per byte are, as long as the corpus is the same one (see the corpus cell).

In [ ]:
!nvidia-smi
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

## Repository

Private repo: put a token in `TOKEN` and the clone URL becomes
`https://{TOKEN}@github.com/whynotramaa/octlm.git`.

In [ ]:
import os, pathlib

REPO = 'https://github.com/whynotramaa/octlm.git'
BRANCH = 'main'
if pathlib.Path('/content/octlm/.git').exists():
    !cd /content/octlm && git fetch origin && git checkout $BRANCH && git pull
else:
    !git clone --branch $BRANCH $REPO /content/octlm
os.chdir('/content/octlm')
!git log --oneline -1

## Corpus

`octlm.corpus` reads the Python standard library of the machine it runs on, so Colab builds a
different corpus than the local machine and the numbers will not line up with `notes/day2.md`.

To reproduce local runs, skip the build below and copy `data/` and `artifacts/day2/` from
Drive instead (next cell).

In [ ]:
!python -m octlm.corpus
!ls -la data artifacts/day2 2>/dev/null

In [ ]:
# Optional: use the corpus and tokenizer built on the local machine.
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p data artifacts/day2
# !cp -r /content/drive/MyDrive/octlm/data/. data/
# !cp -r /content/drive/MyDrive/octlm/artifacts/day2/. artifacts/day2/

## Smoke check

Builds the model and prints the config hash, parameter count, and logits shape.

In [ ]:
!python -m octlm.train --config configs/day2.toml --dry-run
!python -m octlm.bench --dummy --contexts 128 512 2048 --device cuda

## Day 2 training runs

`variants` is the architecture grid, `length` is the context sweep. These are the two stages
worth a GPU. `tiled`, `equivalence`, `sdpa`, and `cache` are CPU measurements that are already
recorded in `notes/day2.md`; the SDPA benchmark measures process RSS, which means nothing on a
GPU, so leave those on CPU.

In [ ]:
!python -m octlm.day2 variants --device cuda

In [ ]:
!python -m octlm.day2 length --config configs/day2-long.toml --device cuda

In [ ]:
!python -m octlm.day2 report

## Day 1 trainer

`--device auto` is the default everywhere, so `--device cuda` is only needed to be explicit or
to force CPU for a comparison.

In [ ]:
!python -m octlm.train --config configs/day2.toml --tokenizer bpe \
    --train data/train.jsonl --validation data/val.jsonl \
    --checkpoint artifacts/day2/colab.pt --metrics runs/colab-day2.jsonl --device cuda

## Keep the results

Colab deletes the VM. `runs/` and `artifacts/` are gitignored, so copy them out.

In [ ]:
!tar czf /content/octlm-results.tar.gz runs artifacts
from google.colab import files
files.download('/content/octlm-results.tar.gz')